# T3 — Introductory Exploratory Data Analysis
Sara Milovanova & Biljana Vitanova

**All aggregations pre-computed by `t3_aggregations.py` (sbatch job) and stored in `data/t3/`.**  
This notebook only loads small CSVs and plots — every cell runs in under 1 second.

Tools: DuckDB + Dask (in the aggregation script), Pandas + Matplotlib (here).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import geopandas as gpd
from pathlib import Path

T3   = Path("/d/hpc/projects/FRI/bigdata/students/sm_bv/final_project/data/t3")
DATA = Path("/d/hpc/projects/FRI/bigdata/students/sm_bv/data")

yearly   = pd.read_csv(T3/"yearly_trips.csv",   index_col=0)
monthly  = pd.read_csv(T3/"monthly_trips.csv",  index_col=0)
hourly   = pd.read_csv(T3/"hourly_trips.csv",   index_col=0)
dow      = pd.read_csv(T3/"dow_trips.csv",      index_col=0)
fare_df  = pd.read_csv(T3/"fare_distance.csv",  index_col=[0,1])
fare_dist= pd.read_csv(T3/"fare_distribution.csv", index_col=0)
top_pu   = pd.read_csv(T3/"top_pu_zones.csv",   index_col=0)
top_do   = pd.read_csv(T3/"top_do_zones.csv",   index_col=0)
company  = pd.read_csv(T3/"fhvhv_company.csv",  index_col=[0,1])
covid    = pd.read_csv(T3/"covid_monthly.csv",  index_col=0, parse_dates=True)

colors = {"yellow":"#FFD700", "green":"#2ca02c", "fhv":"#1f77b4", "fhvhv":"#d62728"}
print("All CSVs loaded.")

---
## 1. Dataset Overview

In [ ]:
overview = pd.DataFrame({
    "total_trips": yearly.sum().map("{:,.0f}".format),
    "first_year":  yearly.apply(lambda s: s.first_valid_index()),
    "last_year":   yearly.apply(lambda s: s.last_valid_index()),
})
overview

---
## 2. Trip Volume by Year

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
for col in yearly.columns:
    s = yearly[col].dropna()
    ax.plot(s.index.astype(int), s/1e6, marker="o", label=col, color=colors[col])
ax.set_title("Annual Trip Volume by Dataset (each from its start year to 2026)", fontsize=13)
ax.set_ylabel("Trips (millions)"); ax.set_xlabel("Year")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"{v:.0f}M"))
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 3. Monthly Seasonality — All Datasets (2019–2026)

In [ ]:
month_labels = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, monthly.columns):
    ax.bar(monthly.index.astype(int), monthly[col]/1e6, color=colors[col], alpha=0.85)
    ax.set_title(f"{col} (2019–2026)"); ax.set_xticks(range(1,13))
    ax.set_xticklabels(month_labels, rotation=45, fontsize=7)
    ax.set_ylabel("Trips (M)"); ax.grid(axis="y", alpha=0.3)
plt.suptitle("Monthly Trip Distribution (2019–2026)", fontsize=13)
plt.tight_layout(); plt.show()

---
## 4. Hourly Distribution (2019–2026)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
for col in hourly.columns:
    pct = hourly[col] / hourly[col].sum() * 100
    ax.plot(hourly.index.astype(int), pct, marker="o", label=col, color=colors[col])
ax.set_title("Hourly Trip Distribution — % of Daily Volume (2019–2026)", fontsize=13)
ax.set_xlabel("Hour"); ax.set_ylabel("Share (%)")
ax.set_xticks(range(24)); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 5. Day-of-Week Patterns (2019–2023)

In [ ]:
day_labels = ["Sun","Mon","Tue","Wed","Thu","Fri","Sat"]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, dow.columns):
    ax.bar(dow.index.astype(int), dow[col]/1e6, color=colors[col], alpha=0.85)
    ax.set_title(col); ax.set_xticks(range(7))
    ax.set_xticklabels(day_labels, rotation=45)
    ax.set_ylabel("Trips (M)"); ax.grid(axis="y", alpha=0.3)
plt.suptitle("Day-of-Week Distribution (2019–2026)", fontsize=13)
plt.tight_layout(); plt.show()

---
## 6. Average Fare & Distance by Year

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ds in ["yellow","green","fhvhv"]:
    if ds not in fare_df.index.get_level_values(0): continue
    sub = fare_df.loc[ds]; sub.index = sub.index.astype(int)
    axes[0].plot(sub.index, sub["avg_fare"], marker="o", label=ds, color=colors[ds])
    axes[1].plot(sub.index, sub["avg_dist"], marker="o", label=ds, color=colors[ds])
axes[0].set_title("Average Fare by Year"); axes[0].set_ylabel("USD")
axes[1].set_title("Average Distance by Year"); axes[1].set_ylabel("Miles")
for ax in axes: ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 7. Fare Distribution — Yellow (2022–2026)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(fare_dist.index.astype(float), fare_dist["cnt"]/1e6,
       width=4.5, color="#FFD700", edgecolor="black", linewidth=0.3)
ax.set_title("Yellow Taxi — Fare Distribution (2022–2026)", fontsize=13)
ax.set_xlabel("Fare (USD, $5 buckets)"); ax.set_ylabel("Trips (M)")
ax.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()

---
## 8. Spatial — Top 10 Pickup & Dropoff Zones (Yellow, 2019–2026)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, df, title in [
    (axes[0], top_pu, "Top 10 Pickup Zones"),
    (axes[1], top_do, "Top 10 Dropoff Zones"),
]:
    ax.barh(df.index.astype(str), df["pickups"]/1e6, color="#FFD700", edgecolor="black")
    ax.set_title(f"Yellow — {title} (2019–2026)")
    ax.set_xlabel("Trips (M)"); ax.invert_yaxis(); ax.grid(axis="x", alpha=0.3)
plt.tight_layout(); plt.show()

---
## 8b. Spatial — Pickups by NYC Borough (All Datasets, 2019–2026)

In [ ]:
zones_gdf = gpd.read_file(str(DATA/"taxi_zones/taxi_zones.shp")).set_index("LocationID").to_crs("EPSG:4326")

borough = {}
for ds in ["yellow","green","fhv","fhvhv"]:
    csv = T3/f"zone_pickups_{ds}.csv"
    if csv.exists():
        counts = pd.read_csv(csv, index_col=0)
        gdf = zones_gdf.join(counts, how="left")
        borough[ds] = gdf.groupby("borough")["pickups"].sum().fillna(0)

borough_df = pd.DataFrame(borough).fillna(0)
borough_df.div(1e6).plot(kind="bar", figsize=(12, 5), colormap="tab10")
plt.title("Pickups by NYC Borough — All Datasets (2019–2026)", fontsize=13)
plt.xlabel("Borough"); plt.ylabel("Trips (M)")
plt.xticks(rotation=30, ha="right"); plt.legend(); plt.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()
print(borough_df.div(1e6).round(2).to_string())

---
## 9. FHVHV — Company Breakdown & Shared Rides

In [ ]:
company_r = company.reset_index()
pivot = company_r.pivot_table(index="year", columns="company", values="trips", fill_value=0)
shared_pct = company_r.groupby("year").apply(
    lambda g: g["shared"].sum() / g["trips"].sum() * 100
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
pivot.div(1e6).plot(kind="bar", ax=axes[0], colormap="Set2")
axes[0].set_title("FHVHV Trips by Company"); axes[0].set_ylabel("Trips (M)")
axes[0].tick_params(axis="x", rotation=0); axes[0].grid(axis="y", alpha=0.3)
axes[1].bar(shared_pct.index.astype(int), shared_pct, color="#d62728", alpha=0.8)
axes[1].set_title("FHVHV Shared Ride Request Rate")
axes[1].set_ylabel("% shared"); axes[1].grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

---
## 10. COVID-19 Impact — Monthly Volume 2019–2022

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for col in covid.columns:
    s = covid[col].dropna()
    ax.plot(pd.to_datetime(s.index), s/1e6, label=col, color=colors[col])
ax.axvline(pd.Timestamp("2020-03-01"), color="red", linestyle="--", alpha=0.7, label="COVID lockdown")
ax.set_title("Monthly Trip Volume 2019–2022", fontsize=13)
ax.set_ylabel("Trips (M)"); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 11. Cross-Dataset Similarity — Hourly & Annual Correlation

In [ ]:
hourly_pct = hourly.div(hourly.sum()) * 100
corr_h = hourly_pct.corr()
yearly_common = yearly.loc[2019:2025].dropna()
corr_y = yearly_common.corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for col in hourly_pct.columns:
    axes[0].plot(hourly_pct.index.astype(int), hourly_pct[col],
                 marker="o", label=col, color=colors[col])
axes[0].set_title("Normalised Hourly Profiles"); axes[0].set_xlabel("Hour")
axes[0].set_ylabel("%"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

im = axes[1].imshow(corr_h.values, vmin=0.8, vmax=1.0, cmap="YlOrRd")
axes[1].set_xticks(range(len(corr_h))); axes[1].set_xticklabels(corr_h.columns, rotation=45)
axes[1].set_yticks(range(len(corr_h))); axes[1].set_yticklabels(corr_h.index)
for i in range(len(corr_h)):
    for j in range(len(corr_h)):
        axes[1].text(j,i,f"{corr_h.values[i,j]:.3f}",ha="center",va="center",fontsize=9)
plt.colorbar(im, ax=axes[1])
axes[1].set_title("Pearson Correlation — Hourly Profiles")
plt.tight_layout(); plt.show()
print("Hourly:\n", corr_h.round(3).to_string())
print("\nAnnual:\n", corr_y.round(3).to_string())

---
## 12. Summary of Key Findings

| Finding | Observation |
|---|---|
| **Yellow decline** | Peak ~170M trips/year 2013–2014; ~30M by 2022 as FHVHV took over |
| **FHVHV dominance** | By 2022 FHVHV carries more trips than all traditional datasets combined |
| **Uber vs Lyft** | Uber (HV0003) ~70–75%; Lyft ~25% consistently |
| **COVID** | March 2020: ~80–90% drop; yellow/green never fully recovered |
| **Hourly similarity** | All datasets correlate r > 0.95 — 8–9 AM, 5–7 PM, 10 PM–2 AM peaks |
| **Seasonality** | Oct–Dec busiest; Feb–Mar slowest |
| **Shared rides** | FHVHV shared requests peaked 2019–2020, dropped post-COVID |
| **Fare trend** | Yellow avg fare ~$12 (2012) → ~$18 (2024) |

---
## 11b. Pairwise Correlation — Monthly, DOW, Hourly & Annual Patterns